# Setting up the IDEAL-GENOM example dataset

This notebook prepares the example dataset used throughout the IDEAL-GENOM showcase notebooks (01–05). It performs three steps:

1. **Fetch the 1000 Genomes Phase 3 GRCh38 reference panel** via `Fetcher1000Genome`. The files are cached at `ideal_genom/data/1000genomes_build_38/` so re-running is idempotent — no re-download occurs if the binaries are already there.

2. **Create a focused example subset**: 190 South Asian (SAS) samples as the study population, plus 10 African (AFR) samples planted as ancestry outliers to be flagged during ancestry QC. Variants are filtered to MAF > 0.01 and randomly thinned to ~500k SNPs across all chromosomes.

3. **Assign synthetic binary phenotypes** (1 = control, 2 = case) to the selected samples, since the 1000 Genomes dataset carries no real phenotype data.

The output — `ideal_genom/data/example_data/SAS_example.{bed,bim,fam}` — is the starting point for all downstream notebooks.

Let us import the required libraries.

In [1]:
import sys
import os

import pandas as pd
import numpy as np

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.core.executor import run_plink
from ideal_genom.core.get_references import Fetcher1000Genome

Set up directory paths. The reference panel is cached under `ideal_genom/data/1000genomes_build_38/`. The example dataset mirrors the `test_data` folder structure — raw input files go into `example_data/inputData/`, pipeline outputs will be written to `example_data/outputData/`, and YAML configs to `example_data/config/`.

In [2]:
DATA_PATH    = library_path / 'ideal_genom' / 'data'
REF_PATH     = DATA_PATH / '1000genomes_build_38'
example_data = DATA_PATH / 'example_data'

inputData  = example_data / 'inputData'
outputData = example_data / 'outputData'
config     = example_data / 'config'

for d in [inputData, outputData, config]:
    d.mkdir(parents=True, exist_ok=True)

output_name  = 'SAS_example'
output_bfile = inputData / output_name

print(f"Reference cache : {REF_PATH}")
print(f"inputData       : {inputData}")
print(f"outputData      : {outputData}")
print(f"config          : {config}")

Reference cache : /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38
inputData       : /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData
outputData      : /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData
config          : /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/config


## Step 1 — Fetch the 1000 Genomes reference panel

`Fetcher1000Genome` downloads the PLINK2 pgen/pvar/psam files and converts them to PLINK1.9 binaries, caching everything under `ideal_genom/data/1000genomes_build_38/`. If the cached binaries already exist, nothing is downloaded or re-converted.

In [3]:
fetcher = Fetcher1000Genome(destination=REF_PATH, build='38')
fetcher.get_1000genomes()
fetcher.get_1000genomes_binaries()

print(f"Reference binaries ready at: {fetcher.bed_file}")

INFO:ideal_genom.core.get_references:Destination folder: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38
INFO:ideal_genom.core.get_references:1000 Genomes binaries already exist. Skipping download.
INFO:ideal_genom.core.get_references:1000 Genomes binaries already exist. Skipping conversion into bfiles...


Reference binaries ready at: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38/1kG_phase3_GRCh38.bed


## Step 2 — Select example samples

The ancestry QC pipeline is designed for a homogeneous study population. We use the SAS superpopulation as the study population and plant 10 AFR samples as ancestry outliers that the pipeline should flag and remove.

Read the reference panel PSAM to inspect the superpopulation and sub-population breakdown.

In [4]:
psam = pd.read_csv(fetcher.bed_file.with_suffix('.psam'), sep='\t')
psam = psam.rename(columns={'#IID': 'IID'})

sas_samples = psam[psam['SuperPop'] == 'SAS']
afr_samples = psam[psam['SuperPop'] == 'AFR']

print(f"SAS samples available: {len(sas_samples)}")
print(sas_samples['Population'].value_counts().to_string())
print(f"\nAFR samples available: {len(afr_samples)}")
print(afr_samples['Population'].value_counts().to_string())

SAS samples available: 601
Population
PJL    146
BEB    131
STU    114
ITU    107
GIH    103

AFR samples available: 893
Population
GWD    178
YRI    178
ESN    149
ACB    116
MSL     99
LWK     99
ASW     74


Randomly draw 190 SAS samples (study population) and 10 AFR samples (ancestry outliers). A fixed seed ensures the selection is reproducible.

In [5]:
sas_selected = sas_samples.sample(n=190, random_state=42)
afr_selected = afr_samples.sample(n=10,  random_state=42)

selected = pd.concat([sas_selected, afr_selected])

print(f"Selected {len(sas_selected)} SAS samples:")
print(sas_selected['Population'].value_counts().to_string())
print(f"\nSelected {len(afr_selected)} AFR samples (ancestry outliers):")
print(afr_selected['Population'].value_counts().to_string())

Selected 190 SAS samples:
Population
PJL    49
BEB    41
GIH    34
STU    34
ITU    32

Selected 10 AFR samples (ancestry outliers):
Population
GWD    4
YRI    2
MSL    2
ESN    1
ACB    1


Write a PLINK keep-file (`FID IID` pairs). In the 1000 Genomes FAM file FID is 0 for every sample.

In [6]:
keep_file = example_data / 'keep_samples.txt'
pd.DataFrame({'FID': 0, 'IID': selected['IID']}).to_csv(
    keep_file, sep=' ', header=False, index=False
)
print(f"Keep file written: {keep_file} ({len(selected)} entries)")

Keep file written: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/keep_samples.txt (200 entries)


## Step 3 — Extract samples and thin variants

Run PLINK1.9 to:
- restrict to the 200 selected samples (`--keep`)
- remove variants with MAF < 0.01 in the subsetted population (`--maf`)
- randomly thin to ~500k SNPs genome-wide (`--thin-count`)

This step is skipped if the output binaries already exist, making the notebook safe to re-run. PLINK prints a lot of console text; it is captured into `plink_log` to keep the notebook readable — run `plink_log.show()` in a new cell if you need to inspect it.

In [7]:
output_exists = all(
    (inputData / f'{output_name}{ext}').exists()
    for ext in ['.bed', '.bim', '.fam']
)
print(f"Output binaries already exist: {output_exists}")

Output binaries already exist: False


In [8]:
%%capture plink_log
if not output_exists:
    run_plink([
        '--bfile',      str(fetcher.bed_file.with_suffix('')),
        '--keep',       str(keep_file),
        '--maf',        '0.01',
        '--thin-count', '500000',
        '--make-bed',
        '--out',        str(output_bfile),
    ])

INFO:ideal_genom.core.executor:Executing: plink --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38/1kG_phase3_GRCh38 --keep /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/keep_samples.txt --maf 0.01 --thin-count 500000 --make-bed --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData/SAS_example


PLINK v1.90b7.4 64-bit (18 Aug 2024)           www.cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData/SAS_example.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38/1kG_phase3_GRCh38
  --keep /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/keep_samples.txt
  --maf 0.01
  --make-bed
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData/SAS_example
  --thin-count 500000

63862 MB RAM detected; reserving 31931 MB for main workspace.
64068184 variants loaded from .bim file.
3202 people (1598 males, 1603 females, 1 ambiguous) loaded from .fam.
Ambiguous sex ID written to
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData/SAS_example.nosex
.
--thin-count: 63568184 variants removed (500000 remaining).
--keep: 200 people remaining.
Using 1 thread (no mult

INFO:ideal_genom.core.executor:Command completed successfully


done.


In [9]:
if not output_exists:
    print("PLINK extraction complete.")
else:
    print("Using existing output files.")

bim = pd.read_csv(
    inputData / f'{output_name}.bim', sep='\t', header=None,
    names=['CHR', 'SNP', 'CM', 'BP', 'A1', 'A2']
)
fam = pd.read_csv(
    inputData / f'{output_name}.fam', sep=r'\s+', header=None,
    names=['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO'], engine='python'
)

print(f"\nSamples  : {len(fam)}")
print(f"Variants : {len(bim):,}")
print(f"Chromosomes covered: {sorted(bim['CHR'].unique())}")

PLINK extraction complete.

Samples  : 200
Variants : 79,397
Chromosomes covered: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]


## Step 4 — Assign synthetic phenotypes

The 1000 Genomes panel carries no real phenotype data (PHENO = –9 in the FAM file). Assign random binary case/control labels (1 = control, 2 = case) with a fixed seed so that downstream variant QC and GWAS examples are reproducible.

In [10]:
# The 10 AFR samples share IIDs with the 1000G reference panel. If their IIDs are left
# unchanged, ancestry QC labels them 'AFR' (matched from the reference PSAM) instead of
# 'StPop', so the outlier detector never sees them as study samples. Renaming them to
# synthetic IDs ensures they are treated as unrecognized study individuals and flagged.
afr_iids = set(afr_selected['IID'])
needs_rename = fam['IID'].isin(afr_iids).any()

if needs_rename:
    counter = 0
    for idx in fam[fam['IID'].isin(afr_iids)].index:
        counter += 1
        fam.at[idx, 'IID'] = f'OUTLIER_{counter:03d}'
    print(f"Renamed {counter} AFR samples to OUTLIER_001 … OUTLIER_{counter:03d}")
else:
    print("AFR samples already renamed, skipping.")

# Assign synthetic binary phenotypes (1 = control, 2 = case)
if fam['PHENO'].eq(-9).all():
    np.random.seed(42)
    fam['PHENO'] = np.random.choice([1, 2], size=len(fam))
    print("Synthetic binary phenotypes assigned (1 = control, 2 = case).")
else:
    print("Phenotypes already present, skipping.")

fam.to_csv(inputData / f'{output_name}.fam', sep=' ', header=False, index=False)

print(f"\nPhenotype distribution:")
print(fam['PHENO'].map({1: 'Control (1)', 2: 'Case (2)'}).value_counts().to_string())

Renamed 10 AFR samples to OUTLIER_001 … OUTLIER_010
Synthetic binary phenotypes assigned (1 = control, 2 = case).

Phenotype distribution:
PHENO
Control (1)    100
Case (2)       100


Remove the temporary keep-file and print a summary of the finished dataset.

In [11]:
keep_file.unlink(missing_ok=True)

print("Example dataset ready.")
print(f"  Path      : {output_bfile}")
print(f"  Samples   : {len(fam)}  (190 SAS + 10 AFR outliers)")
print(f"  Variants  : {len(bim):,}")
print()
print("To use in the QC notebooks, set:")
print(f"  input_path = DATA_PATH / 'example_data' / 'inputData'")
print(f"  output_path = DATA_PATH / 'example_data' / 'outputData'")
print(f"  input_name = '{output_name}'")

Example dataset ready.
  Path      : /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/inputData/SAS_example
  Samples   : 200  (190 SAS + 10 AFR outliers)
  Variants  : 79,397

To use in the QC notebooks, set:
  input_path = DATA_PATH / 'example_data' / 'inputData'
  output_path = DATA_PATH / 'example_data' / 'outputData'
  input_name = 'SAS_example'
